## Launching colab kernel

8-chip
```
colab launch  //third_party/py/torchtitan:torchtitan_colab \
 --xm_resource_alloc=cloud-dynamic/cmcs-xm \
 --accelerator=vl:4x2  \
 --label="$USER - vl4x2 `date '+%Y-%m-%d %H:%M:%S'`"
```

if not able to get vl chips, change to vlp chips
```
colab launch  //third_party/py/torchtitan:torchtitan_colab \
 --xm_resource_alloc=cloud-dynamic/cmcs-xm \
 --accelerator=vlp:2x4  \
 --label="$USER - vlp2x4 `date '+%Y-%m-%d %H:%M:%S'`"
```
and need to change `third_party/py/torch_tpu/_internal/distributed/tpu_env.py` with
```
-    flags.FLAGS.deepsea_host_bounds = "4,2,1"  # 8 TPUs total
+    flags.FLAGS.deepsea_host_bounds = "2,4,1"  # 8 TPUs total
```

In [1]:
%%writefile /tmp/debug_model.toml
[job]
dump_folder = "./outputs"
description = "Qwen 3 debug model training"

[profiling]
enable_profiling = false
save_traces_folder = "profile_trace"
profile_freq = 100

[metrics]
log_freq = 1
enable_tensorboard = false
save_tb_folder = "tb"

[model]
name = "qwen3"
flavor = "debugmodel"
hf_assets_path = "./assets/hf/Qwen3-0.6B"
# converters = ["float8"]

[optimizer]
name = "AdamW"
lr = 3e-4
eps = 1e-8
implementation = "foreach"  # TODO: b/45811390 - Remove this once fused optimizer is supported on TPU.


[lr_scheduler]
warmup_steps = 2  # lr scheduler warm up, 20% total steps

[training]
local_batch_size = 4
seq_len = 128
max_norm = 1.0  # grad norm clipping
steps = 10
dataset = "c4_test"  # supported datasets: c4_test (2K), c4 (177M)

[parallelism]
data_parallel_replicate_degree = 1
data_parallel_shard_degree = -1
fsdp_reshard_after_forward = "default" # default / never / always
tensor_parallel_degree = 1
context_parallel_degree = 1

[checkpoint]
enable = false
folder = "checkpoint"
interval = 500
last_save_model_only = false
export_dtype = "float16"
async_mode = "disabled" # ["disabled", "async", "async_with_pinned_mem"]

[activation_checkpoint]
mode = "selective"  # ["none", "selective", "full"]
selective_ac_option = "op"  # "int" = ac every positive int layer or 'op', ac based on ops policy

[compile]
enable=false
components = ["model", "loss"]

[quantize.linear.float8]
enable_fsdp_float8_all_gather = false
precompute_float8_dynamic_scale_for_fsdp = false
filter_fqns = ["output"]


In [2]:
import torchtitan.experiments.tpu.train

import torchtitan.config
from torchtitan.experiments import tpu
import os
import datetime
from torchtitan.experiments.tpu import tpu_job_config as tpu_job_config_module


num_devices = 8
accelerator_device_type=tpu.accelerator_device_type.AcceleratorDeviceType.TPU

config_manager = torchtitan.config.ConfigManager(tpu_job_config_module.TPUJobConfig)
config = config_manager.parse_args([
    "--job.config_file=/tmp/debug_model.toml",
    "--model.name=qwen3_tpu",
    "--model.hf_assets_path=/cns/is-d/home/torch-tpu-xm/torchtitan/tests/assets/tokenizer",
    "--training.dataset_path=/cns/is-d/home/torch-tpu-xm/torchtitan/tests/assets/c4_test",
    "--training.seq_len=128",
    "--training.dataset=c4_test",
    "--parallelism.data_parallel_shard_degree=1",
    f"--parallelism.tensor_parallel_degree={num_devices}",
    "--tpu_config.use_fairscale"
])
config.training.steps = 20

# enable profiling
config.profiling.enable_profiling=True
output_dir = os.path.join("/tmp", "xprof_" + datetime.datetime.now().strftime("%Y%m%d_%H-%M-%S"))
config.profiling.save_traces_folder=output_dir


for k, v in config.__dict__.items():
  print(f"{k} -> {v}\n")

In [3]:
from torchtitan.experiments.tpu import distributed as torchtitan_distributed

torchtitan_distributed.run_distributed(
    num_devices,
    accelerator_device_type,
    torchtitan.experiments.tpu.train.start_trainer,
    config,
    )

print("done")

In [4]:
from torchtitan.experiments.tpu.profiling import upload_xprof_xplane_pb_files_from_dir

upload_xprof_xplane_pb_files_from_dir(output_dir, merge="all")